# Foodie Advisor · 1. Datos, exploración y línea clásica en español

Este notebook reúne las fases `00_datos`, `01_eda`, `02_semeval_clasico`. Cada sección escribe su `resultados/hallazgos_XX.json`, que es lo que lee la app.

## 00 · Datos: carga, limpieza y división en oraciones

| | |
|---|---|
| **Qué entra** | XML de SemEval-2016 ES (`data/`) y los 6 CSV de TripAdvisor (caché de `kagglehub`) |
| **Qué sale** | `data/procesado/{ciudad}_reseñas.parquet`, `{ciudad}_oraciones.parquet`, `semeval.parquet` |

Este notebook no modela nada: deja los datos listos para los siguientes y comprueba que, con el código movido al paquete `foodie_advisor`, se reproducen las cifras de Madrid del avance (154,819 reseñas, F1 0.834, −1.58★).

In [1]:
import sys, time
sys.path.insert(0, "..")

import pandas as pd
from foodie_advisor import CIUDADES, PROCESADO, datos, lexico, costos, modelos

pd.set_option("display.width", 140)

### 1. SemEval-2016, restaurantes en español

Una fila por opinión anotada. Quitamos el texto sobrante del formulario ("Escriba aquí su comentario"), igual que en el avance.

In [2]:
sem = datos.cargar_semeval()
op = sem.dropna(subset=["cat"])
pd.Series({
    "Reseñas": sem.rid.nunique(),
    "Oraciones": sem.sid.nunique(),
    "Opiniones anotadas": len(op),
    "Opiniones implícitas (target NULL)": int((op.target == "NULL").sum()),
    "Categorías de aspecto": op.cat.nunique(),
}, name="SemEval-2016 ES").to_frame()

,SemEval-2016 ES
Reseñas,895
Oraciones,2946
Opiniones anotadas,3792
Opiniones implícitas (target NULL),1124
Categorías de aspecto,12


### 2. TripAdvisor: las 6 ciudades

Aplicamos a cada ciudad los mismos filtros del avance: estrellas, fecha y texto presentes; sin textos duplicados; al menos 5 palabras; restaurantes con 20 reseñas o más. El CSV de New Delhi trae filas malformadas, así que se lee con el parser de Python y `on_bad_lines="skip"`.

La primera ejecución tarda unos minutos (London pesa 940 MB). Después lee los parquet ya generados.

In [3]:
t0 = time.time()
pasos = datos.procesar_todo()
print(f"Listo en {time.time() - t0:.0f} s")
pasos

Listo en 0 s


,Madrid,Barcelona,London,Paris,New_York,New_Delhi
Paso,,,,,,
Reseñas en el CSV,176848,416356,996567,508202,510463,147602
"Con estrellas, fecha y texto",176848,416354,996546,508200,510461,147598
Estrellas entre 1 y 5,176848,416354,996546,508200,510461,147598
Sin textos duplicados,176783,416185,996253,508020,510341,147444
Con 5 palabras o más,176749,416121,996153,507948,510184,147418
En restaurantes con 20 reseñas o más,154819,392568,995832,465201,509058,128838


In [4]:
resumen = []
for c in CIUDADES:
    r = datos.cargar_resenas(c, ["restaurante", "estrellas", "fecha", "autor", "palabras"])
    n_rest = r.groupby("restaurante").size()
    por_autor = r.groupby("autor").size()
    resumen.append({
        "Ciudad": c, "Reseñas": len(r), "Restaurantes": n_rest.size, "Rest. ≥100": int((n_rest >= 100).sum()),
        "% 1-2★": round(r.estrellas.le(2).mean() * 100, 1), "% 5★": round(r.estrellas.eq(5).mean() * 100, 1),
        "Mediana palabras": int(r.palabras.median()),
        "% de autores con >1 reseña": round(por_autor.gt(1).mean() * 100, 1),
        "% de reseñas de autores con >1": round(r.autor.map(por_autor).gt(1).mean() * 100, 1),
        "Desde": r.fecha.min().year, "Hasta": r.fecha.max().year,
    })
resumen = pd.DataFrame(resumen).set_index("Ciudad")
print(f"Total: {resumen['Reseñas'].sum():,} reseñas de {resumen.Restaurantes.sum():,} restaurantes "
      f"({resumen['Rest. ≥100'].sum():,} con 100 reseñas o más)")
resumen

Total: 2,646,316 reseñas de 11,857 restaurantes (5,546 con 100 reseñas o más)


,Reseñas,Restaurantes,Rest. ≥100,% 1-2★,% 5★,Mediana palabras,% de autores con >1 reseña,% de reseñas de autores con >1,Desde,Hasta
Ciudad,,,,,,,,,,
Madrid,154819,1414,366,9.0,56.3,52,30.5,58.5,2007,2020
Barcelona,392568,2430,914,10.6,58.3,53,31.9,61.3,2006,2020
London,995832,1796,1679,7.8,58.7,63,29.1,64.3,2004,2020
Paris,465201,3975,1191,11.5,50.3,60,33.9,65.5,2003,2020
New_York,509058,1250,1086,7.1,50.7,61,36.0,68.6,2003,2020
New_Delhi,128838,992,310,6.0,51.3,49,29.1,65.3,2007,2020


### 3. De reseñas a oraciones

El detector de fallas trabaja por oración: una reseña puede quejarse del cobro y elogiar la comida en la misma línea, y a nivel de reseña esas dos cosas se mezclan. Cortamos tras `.`, `!`, `?` y en los saltos de párrafo; en el CSV de Kaggle esos saltos quedaron como dos o más espacios seguidos. Descartamos fragmentos de menos de 3 palabras.

In [5]:
ejemplo = datos.cargar_resenas("Madrid", ["review_id", "estrellas", "texto"]).iloc[0]
print(f"Reseña ({ejemplo.estrellas}★):", ejemplo.texto[:400], "\n")
for i, o in enumerate(datos.dividir_oraciones(ejemplo.texto)):
    print(f"  [{i}] {o}")

Reseña (1★): Check your bill when you cancel just in case you get an extra charge surprise on it   There are no more words to describe my experience when I was there. Sad and poor experience!! Never return! Never again 

  [0] Check your bill when you cancel just in case you get an extra charge surprise on it
  [1] There are no more words to describe my experience when I was there.
  [2] Sad and poor experience!!


In [6]:
filas = []
for c in CIUDADES:
    o = pd.read_parquet(datos.ruta_oraciones(c))
    largo = o.oracion.str.split().str.len()
    filas.append({"Ciudad": c, "Oraciones": len(o), "Por reseña (media)": round(o.groupby("review_id").size().mean(), 1),
                  "Palabras por oración (mediana)": int(largo.median()),
                  "Oraciones de 60+ palabras %": round(largo.ge(60).mean() * 100, 2)})
orac_resumen = pd.DataFrame(filas).set_index("Ciudad")
print(f"Total: {orac_resumen.Oraciones.sum():,} oraciones")
orac_resumen

Total: 14,656,074 oraciones


,Oraciones,Por reseña (media),Palabras por oración (mediana),Oraciones de 60+ palabras %
Ciudad,,,,
Madrid,771732,5.0,12,0.24
Barcelona,1981854,5.0,12,0.29
London,5600906,5.6,13,0.45
Paris,2612293,5.6,12,0.28
New_York,3015122,5.9,12,0.23
New_Delhi,674167,5.2,12,0.49


Las oraciones de 60 palabras o más son reseñas casi sin puntuación que el separador no pudo cortar. Son pocas, y el clasificador las recorta a un largo máximo.

### 4. Verificación: ¿se reproducen las cifras del avance en Madrid?

Antes de cambiar nada comprobamos que el código del paquete da los mismos números que el notebook del avance (`notebooks/avances/`). Si no coinciden, cualquier diferencia posterior sería imposible de atribuir.

In [7]:
madrid = datos.cargar_resenas("Madrid")
pol = modelos.polaridad_semeval(sem)
X = lexico.matriz_terminos(madrid.texto)
F = lexico.fallas_por_etapa(X)
beta, lo, hi = costos.efectos_fijos(F.values, madrid.estrellas.values, madrid.restaurante.values, reps=200)
costo = pd.DataFrame({"β dentro del restaurante": beta, "IC95% inf": lo, "IC95% sup": hi}, index=F.columns).round(2)
costo

,β dentro del restaurante,IC95% inf,IC95% sup
1. Llegada y mesa,-0.20,-0.28,-0.14
2. Atención en sala,-1.58,-1.63,-1.53
3. Cocina → mesa,-0.75,-0.80,-0.70
4. Cobro,-0.98,-1.09,-0.86
Fuera del ciclo: cocina,-1.18,-1.25,-1.13
Fuera del ciclo: precio,-0.62,-0.67,-0.57
Fuera del ciclo: local,-0.39,-0.45,-0.34


In [8]:
avance = {
    "Reseñas Madrid": (154_819, len(madrid)),
    "Restaurantes Madrid": (1_414, madrid.restaurante.nunique()),
    "F1 polaridad LogReg (SemEval)": (0.834, round(pol["f1_logreg"], 3)),
    "F1 polaridad NB (SemEval)": (0.780, round(pol["f1_nb"], 3)),
    "Costo Atención ★": (-1.58, costo.loc["2. Atención en sala", "β dentro del restaurante"]),
    "Costo Cobro ★": (-0.98, costo.loc["4. Cobro", "β dentro del restaurante"]),
    "Costo Cocina → mesa ★": (-0.75, costo.loc["3. Cocina → mesa", "β dentro del restaurante"]),
    "Costo Llegada ★": (-0.20, costo.loc["1. Llegada y mesa", "β dentro del restaurante"]),
}
check = pd.DataFrame(avance, index=["avance", "ahora"]).T
check["coincide"] = check.avance.eq(check.ahora)
assert check.coincide.all(), check[~check.coincide]
check

,avance,ahora,coincide
Reseñas Madrid,154819.000,154819.000,True
Restaurantes Madrid,1414.000,1414.000,True
F1 polaridad LogReg (SemEval),0.834,0.834,True
F1 polaridad NB (SemEval),0.780,0.780,True
Costo Atención ★,-1.580,-1.580,True
Costo Cobro ★,-0.980,-0.980,True
Costo Cocina → mesa ★,-0.750,-0.750,True
Costo Llegada ★,-0.200,-0.200,True


### 5. SemEval a parquet y archivos generados

In [9]:
for p in sorted(PROCESADO.glob("*.parquet")):
    print(f"{p.name:32} {p.stat().st_size / 1e6:8.1f} MB")

Barcelona_oraciones.parquet         101.7 MB
Barcelona_reseñas.parquet           106.1 MB
London_oraciones.parquet            310.7 MB
London_reseñas.parquet              319.8 MB
Madrid_oraciones.parquet             39.4 MB
Madrid_reseñas.parquet               41.4 MB
New_Delhi_oraciones.parquet          35.7 MB
New_Delhi_reseñas.parquet            37.2 MB
New_York_oraciones.parquet          151.5 MB
New_York_reseñas.parquet            156.4 MB
Paris_oraciones.parquet             137.9 MB
Paris_reseñas.parquet               142.9 MB
pred_regex_v1_Barcelona.parquet       6.7 MB
pred_regex_v1_Madrid.parquet          2.6 MB
semeval.parquet                       0.2 MB


### Cifras clave


In [10]:
from foodie_advisor import resultados
resultados.guardar("00", {
    "resenas_total": int(resumen["Reseñas"].sum()), "restaurantes_total": int(resumen.Restaurantes.sum()),
    "restaurantes_100": int(resumen["Rest. ≥100"].sum()), "oraciones_total": int(orac_resumen.Oraciones.sum()),
    "resenas_crudas_total": int(pasos.iloc[0].sum()),
    "por_ciudad": resumen.drop(columns=["Desde", "Hasta"]), "oraciones_por_ciudad": orac_resumen.Oraciones,
    "pct_resenas_autor_repetido_min": resumen["% de reseñas de autores con >1"].min(),
    "pct_resenas_autor_repetido_max": resumen["% de reseñas de autores con >1"].max(),
    "semeval": {"resenas": int(sem.rid.nunique()), "oraciones": int(sem.sid.nunique()), "opiniones": len(op),
                "implicitas": int((op.target == "NULL").sum())},
    "verificacion_avance": check.ahora, "verificacion_ok": bool(check.coincide.all()),
})

hallazgos_00.json: 12 claves → oraciones_por_ciudad, oraciones_total, pct_resenas_autor_repetido_max, pct_resenas_autor_repetido_min, por_ciudad, resenas_crudas_total, resenas_total, restaurantes_100, restaurantes_total, semeval, verificacion_avance, verificacion_ok


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_00.json')

### Hallazgos

- **Se reproduce el avance.** Con el código en `foodie_advisor/`, Madrid vuelve a dar 154,819 reseñas de 1,414 restaurantes, F1 de polaridad 0.834 (LogReg) y 0.780 (NB) en SemEval, y los mismos costos por etapa (Atención −1.58★, Cobro −0.98★, Cocina → mesa −0.75★, Llegada −0.20★). El `assert` de la sección 4 falla si algo se mueve.
- **Seis ciudades, 2,646,316 reseñas limpias** de 11,857 restaurantes, 5,546 con 100 reseñas o más. London aporta más de un tercio (995,832).
- **La forma de los datos se repite en todas las ciudades:** entre 50% y 59% de reseñas de 5★ y solo 6-12% de 1-2★. Las quejas son escasas en todas partes.
- **14.7 M oraciones,** unas 5 por reseña con mediana de 12 palabras. Es la unidad del clasificador.
- **Los autores se repiten:** entre 58% y 69% de las reseñas viene de autores con más de una. Para el R² y los modelos hay que partir también por autor, no solo por restaurante.

## 01 · Exploración: ¿se repite el patrón de Madrid en las 6 ciudades?

| | |
|---|---|
| **Qué entra** | `data/procesado/{ciudad}_reseñas.parquet` (del 00) y SemEval |
| **Qué sale** | figuras `figuras/01_*.png` |

En el avance solo miramos Madrid. Antes de entrenar nada queremos saber si lo que vimos ahí (pocas quejas, cola larga de restaurantes, la atención como lo que separa las malas reseñas de las buenas) es propio de Madrid o se repite en las otras cinco ciudades.

In [1]:
import collections
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from foodie_advisor import CIUDADES, FIGURAS, datos, lexico, etiquetas as et

sns.set_theme(style="whitegrid", font_scale=0.95)
plt.rcParams["font.family"] = "DejaVu Sans"  # Arial no trae el carácter ★
AZUL, ROJO, GRIS = "#3b6ea5", "#b5483a", "#8c8c8c"
COLOR_CIUDAD = dict(zip(CIUDADES, ["#b5483a", "#3b6ea5", "#5b8c5a", "#e3a33b", "#7a5195", "#8c8c8c"]))
FIGURAS.mkdir(exist_ok=True)

res = {c: datos.cargar_resenas(c, ["restaurante", "estrellas", "fecha", "palabras"]) for c in CIUDADES}
print({c: f"{len(r):,}" for c, r in res.items()})

{'Madrid': '154,819', 'Barcelona': '392,568', 'London': '995,832', 'Paris': '465,201', 'New_York': '509,058', 'New_Delhi': '128,838'}


### 1. Calificaciones: las quejas son escasas en todas partes

In [2]:
dist = pd.DataFrame({c: r.estrellas.value_counts(normalize=True).sort_index() * 100 for c, r in res.items()}).T
fig, ax = plt.subplots(figsize=(8, 3.6))
colores = [ROJO, "#d98a7e", "#d0d0d0", "#9db7d5", AZUL]
dist.iloc[::-1].plot.barh(stacked=True, ax=ax, color=colores, width=0.75)
ax.set(xlabel="% de reseñas", ylabel="", title="Distribución de estrellas por ciudad")
ax.legend([f"{k}★" for k in range(1, 6)], ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.18), frameon=False)
plt.tight_layout(); plt.savefig(FIGURAS / "01_estrellas_ciudad.png", dpi=150); plt.show()
dist.round(1)

<Figure size 800x360 with 1 Axes>

estrellas,1,2,3,4,5
Madrid,4.9,4.1,8.6,26.1,56.3
Barcelona,6.2,4.4,7.9,23.2,58.3
London,3.6,4.2,8.7,24.8,58.7
Paris,6.6,4.9,9.9,28.3,50.3
New_York,3.1,4.0,10.6,31.5,50.7
New_Delhi,2.9,3.0,10.2,32.5,51.3


En todas las ciudades, entre el 50% y el 59% de las reseñas son de 5★, y las de 1-2★ van del 5.9% (New Delhi) al 11.5% (Paris). Las quejas son escasas en todas partes. Eso tiene dos consecuencias: cada queja dice mucho, y cualquier clasificador entrenado sin balancear aprende a decir "no hay falla" y acierta casi siempre. Por eso la escalera del 05 parte de la clase mayoritaria como línea base.

### 2. Cola larga: pocos restaurantes concentran las reseñas

Para comparar restaurantes y estimar sus tasas de falla hace falta volumen. Por eso el scorecard se limita a los que tienen 100 reseñas o más.

In [3]:
fig, ax = plt.subplots(figsize=(6.5, 4))
cola = []
for c, r in res.items():
    n = r.groupby("restaurante").size().sort_values(ascending=False)
    ax.plot(np.arange(1, len(n) + 1), n.values, label=c, color=COLOR_CIUDAD[c])
    cola.append({"Ciudad": c, "Restaurantes": len(n), "Mediana de reseñas": int(n.median()),
                 "Con 100+": int((n >= 100).sum()), "% de reseñas en el 10% más reseñado": round(n.head(len(n) // 10).sum() / n.sum() * 100, 1)})
ax.set(xscale="log", yscale="log", xlabel="Restaurante (rango)", ylabel="Reseñas", title="Reseñas por restaurante")
ax.legend(fontsize=8); plt.tight_layout(); plt.savefig(FIGURAS / "01_cola_larga.png", dpi=150); plt.show()
pd.DataFrame(cola).set_index("Ciudad")

<Figure size 650x400 with 1 Axes>

,Restaurantes,Mediana de reseñas,Con 100+,% de reseñas en el 10% más reseñado
Ciudad,,,,
Madrid,1414,53,366,48.9
Barcelona,2430,67,914,49.2
London,1796,358,1679,34.9
Paris,3975,55,1191,46.8
New_York,1250,240,1086,39.1
New_Delhi,992,54,310,49.8


### 3. Las reseñas malas son más largas

In [4]:
largo = pd.DataFrame({c: r.groupby("estrellas").palabras.median() for c, r in res.items()}).T
largo.columns = [f"{k}★" for k in largo.columns]
largo["Spearman estrellas–palabras"] = [round(spearmanr(r.estrellas, r.palabras)[0], 3) for r in res.values()]
largo

,1★,2★,3★,4★,5★,Spearman estrellas–palabras
Madrid,75.0,74.0,60.0,52.0,48.0,-0.142
Barcelona,76.0,77.0,64.0,53.0,49.0,-0.151
London,107.0,110.0,89.0,64.0,56.0,-0.217
Paris,83.0,81.0,66.0,57.0,57.0,-0.108
New_York,101.0,96.0,74.0,60.0,57.0,-0.145
New_Delhi,78.0,77.0,58.0,50.0,44.0,-0.161


En las 6 ciudades, las reseñas malas son más largas: la mediana de una de 1★ va de 75 a 107 palabras, contra 44 a 57 en las de 5★ (Spearman entre −0.11 y −0.22). Quien se queja cuenta qué pasó, y por eso trabajamos por oración: una reseña de 1★ trae varias oraciones y cada una puede hablar de una etapa distinta.

### 4. Tiempo: de qué años son las reseñas

In [5]:
anual = pd.DataFrame({c: r.fecha.dt.year.value_counts(normalize=True).sort_index() * 100 for c, r in res.items()}).fillna(0)
fig, ax = plt.subplots(figsize=(7.5, 3.6))
for c in CIUDADES:
    ax.plot(anual.index, anual[c], marker="o", ms=3, label=c, color=COLOR_CIUDAD[c])
ax.set(xlim=(2008, 2020.5), xlabel="", ylabel="% de las reseñas de la ciudad", title="Reseñas por año")
ax.legend(fontsize=8, ncol=3); plt.tight_layout(); plt.savefig(FIGURAS / "01_anios.png", dpi=150); plt.show()
print("Año con más reseñas:", anual.idxmax().to_dict())
print("% de reseñas de 2020 (pandemia):", anual.loc[2020].round(1).to_dict())

<Figure size 750x360 with 1 Axes>

Año con más reseñas: {'Madrid': 2017, 'Barcelona': 2016, 'London': 2016, 'Paris': 2016, 'New_York': 2016, 'New_Delhi': 2018}
% de reseñas de 2020 (pandemia): {'Madrid': 2.9, 'Barcelona': 2.4, 'London': 4.6, 'Paris': 2.2, 'New_York': 1.8, 'New_Delhi': 4.6}


### 5. Vocabulario: Zipf y tipos/tokens

La razón tipos/tokens baja a medida que crece el corpus, así que comparamos submuestras con el mismo número de tokens. Cada ciudad aporta tantos tokens como tiene SemEval completo.

In [6]:
sem = datos.cargar_semeval().drop_duplicates("sid")
textos = {"SemEval (es)": sem.texto}
n_sem = sum(len(lexico.tokenizar(x)) for x in sem.texto)
for c in CIUDADES:
    mezcla = datos.cargar_resenas(c, ["texto"]).texto.sample(50_000, random_state=0)
    textos[c] = mezcla[mezcla.str.split().str.len().cumsum() <= n_sem]

filas, frecs = [], {}
for nombre, tx in textos.items():
    toks = [w for x in tx for w in lexico.tokenizar(x)]
    frecs[nombre] = np.array(sorted(collections.Counter(toks).values(), reverse=True))
    filas.append({"Corpus": nombre, "Tokens": len(toks), "Tipos": len(frecs[nombre]), "Tipos/Tokens": round(len(frecs[nombre]) / len(toks), 3),
                  "% hapax": round((frecs[nombre] == 1).mean() * 100, 1)})
display(pd.DataFrame(filas).set_index("Corpus"))

fig, ax = plt.subplots(figsize=(6, 4))
for nombre, f in frecs.items():
    ax.loglog(np.arange(1, len(f) + 1), f, label=nombre, color=COLOR_CIUDAD.get(nombre, "black"), lw=1.2 if nombre in COLOR_CIUDAD else 2)
ax.set(title="Rango-frecuencia (Zipf), mismo número de tokens", xlabel="Rango", ylabel="Frecuencia"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGURAS / "01_zipf.png", dpi=150); plt.show()

,Tokens,Tipos,Tipos/Tokens,% hapax
Corpus,,,,
SemEval (es),42237,5716,0.135,57.9
Madrid,41575,4509,0.108,52.5
Barcelona,41890,4270,0.102,51.8
London,41721,4496,0.108,52.3
Paris,41804,4580,0.110,52.7
New_York,41628,4535,0.109,51.0
New_Delhi,41929,5103,0.122,51.8


<Figure size 600x400 with 1 Axes>

Con el mismo número de tokens, las seis ciudades tienen un vocabulario casi idéntico (tipos/tokens entre 0.102 y 0.110; New Delhi algo más, con 0.122) y menor que el de SemEval (0.135). Las reseñas en inglés de TripAdvisor repiten vocabulario entre ciudades. Eso es lo que permite entrenar en Madrid y aplicar el modelo en Londres o Nueva York sin reentrenar.

### 6. El patrón del avance, ciudad por ciudad

Para cada ciudad medimos, con el regex del avance a nivel de reseña, qué parte de las reseñas de 1-2★ y de 5★ menciona una falla de cada etapa. Es una mirada preliminar: el regex solo ve quejas explícitas, y el 07 repite esto con el clasificador validado.

In [7]:
filas = []
for c in CIUDADES:
    r = datos.cargar_resenas(c, ["estrellas", "texto"])
    Y = et.etiquetas_regex(r.texto)
    for e in et.ETIQUETAS:
        neg, cinco = Y.loc[r.estrellas.values <= 2, e].mean() * 100, Y.loc[r.estrellas.values == 5, e].mean() * 100
        filas.append({"ciudad": c, "etapa": e, "% en 1-2★": neg, "% en 5★": cinco, "veces más en 1-2★": neg / cinco})
patron = pd.DataFrame(filas)
tabla = patron.pivot(index="etapa", columns="ciudad", values="veces más en 1-2★").loc[et.ETIQUETAS, CIUDADES]

fig, ax = plt.subplots(figsize=(8, 3.8))
sns.heatmap(tabla, annot=True, fmt=".0f", cmap="Reds", cbar_kws={"label": "veces más frecuente en 1-2★ que en 5★"}, ax=ax)
ax.set(xlabel="", ylabel="", title="¿Cuánto más aparece cada falla en reseñas malas? (regex, por reseña)")
plt.tight_layout(); plt.savefig(FIGURAS / "01_patron_ciudades.png", dpi=150); plt.show()
patron[patron.etapa == "atencion"].set_index("ciudad").drop(columns="etapa").round(2)

<Figure size 800x380 with 2 Axes>

,% en 1-2★,% en 5★,veces más en 1-2★
ciudad,,,
Madrid,17.21,0.45,38.26
Barcelona,17.19,0.50,34.22
London,16.46,0.62,26.69
Paris,20.78,0.79,26.26
New_York,14.63,0.60,24.49
New_Delhi,10.22,0.39,26.16


El patrón del avance se repite. En las 6 ciudades, la queja de atención es entre 24 y 38 veces más frecuente en reseñas de 1-2★ que en las de 5★, la razón más alta de todas las etapas, seguida por cobro (12 a 30) y sabor (16 a 23). Llegada casi no separa (1 a 2 veces): "queue" y "waited for a table" aparecen también en reseñas de 5★ (una reseña de 5★ de Madrid: *"this place is busy but well worth a short wait for a table"*). Esperar no siempre es una falla, y el 03 lo tiene en cuenta.

**Corrección al avance:** ahí dijimos que la queja de atención era "43 veces" más frecuente en Madrid. Esa cifra salió de dividir porcentajes ya redondeados (17.2 / 0.4). Sin redondear es 17.21% / 0.45% = **38 veces**. Además, la razón es en parte circular, porque el léxico contiene palabras negativas; el 07 la compara contra la tasa de "cualquier oración negativa".

### Cifras clave


In [8]:
res_ttr = pd.DataFrame({"Tipos/Tokens": {k: len(f) / f.sum() for k, f in frecs.items()}})
atn = patron[patron.etapa == "atencion"].set_index("ciudad")
resultados.guardar("01", {
    "pct_5_min": dist[5].min(), "pct_5_max": dist[5].max(),
    "pct_12": (dist[1] + dist[2]), "mediana_palabras": largo,
    "ttr": res_ttr["Tipos/Tokens"], "razon_atencion": atn["veces más en 1-2★"],
    "pct_atencion_12": atn["% en 1-2★"], "pct_atencion_5": atn["% en 5★"],
    "razon_por_etapa": tabla, "atencion_madrid_avance_dicho": 43,
})

hallazgos_01.json: 10 claves → atencion_madrid_avance_dicho, mediana_palabras, pct_12, pct_5_max, pct_5_min, pct_atencion_12, pct_atencion_5, razon_atencion, razon_por_etapa, ttr


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_01.json')

### Hallazgos

- **2,646,316 reseñas en 6 ciudades con la misma forma:** entre 50% y 59% de 5★ y entre 6% y 12% de 1-2★; el 10% de restaurantes más reseñados concentra entre el 35% y el 50% de las reseñas.
- **Las reseñas malas son más largas en todas partes** (mediana de 75 a 107 palabras en 1★, contra 44 a 57 en 5★): conviene clasificar por oración y no por reseña.
- **El vocabulario es casi el mismo entre ciudades** (tipos/tokens entre 0.102 y 0.122 con el mismo número de tokens), lo que justifica entrenar en Madrid y aplicar en el resto.
- **La atención separa las malas reseñas de las buenas en las 6 ciudades:** es entre 24 y 38 veces más frecuente en 1-2★ que en 5★. En Madrid es **38×, no 43× como dijimos en el avance** (redondeo).
- **Llegada casi no separa (1-2×),** porque "queue" aparece también en reseñas buenas. Esperar no es necesariamente una falla.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [9]:
citas = ['this place is busy but well worth a short wait for a table']
verificadas = datos.verificar_citas(citas, ('Madrid', 'Barcelona', 'London', 'Paris', 'New_York', 'New_Delhi'))
print(f"{len(verificadas)} citas verificadas:", verificadas)

1 citas verificadas: {'this place is busy but well worth a short wait for a table': 'Madrid'}


## 02 · La línea clásica en español (SemEval-2016)

| | |
|---|---|
| **Qué entra** | `data/procesado/semeval.parquet` (del 00), vectores fastText SUC |
| **Qué sale** | métricas de polaridad y de aspectos contra la anotación manual de SemEval; figuras `figuras/02_*.png` |

SemEval es el único corpus del proyecto anotado a mano (por los organizadores de la tarea) con aspecto y polaridad. Aquí sirve para medir, contra esa anotación, las decisiones de preprocesamiento que después usamos en inglés. Es un capítulo aparte: no se traduce, y no es un puente hacia TripAdvisor.

In [1]:
import os
from sklearn.metrics import confusion_matrix
from foodie_advisor import FIGURAS, RAIZ, datos, lexico, modelos

sem = pd.read_parquet("../data/procesado/semeval.parquet")
op = sem.dropna(subset=["cat"])

### 1. De qué hablan las opiniones y con qué polaridad

In [2]:
pol = pd.crosstab(op.cat, op.pol.where(op.pol != "conflict"))[["positive", "neutral", "negative"]]
pol["total"] = pol.sum(axis=1)
pol["% negativas"] = (pol.negative / pol.total * 100).round(1)

fig, ax = plt.subplots(figsize=(8, 4.2))
orden = pol.sort_values("% negativas").index
pol.loc[orden, ["positive", "neutral", "negative"]].div(pol.loc[orden, "total"], axis=0).plot.barh(
    stacked=True, ax=ax, color=[AZUL, "#d0d0d0", ROJO], width=0.8)
ax.set(title="SemEval: polaridad dentro de cada categoría", xlabel="Proporción", ylabel="")
ax.legend(["positiva", "neutral", "negativa"], loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
plt.tight_layout(); plt.savefig(FIGURAS / "02_polaridad_categoria.png", dpi=150); plt.show()

por_sid = op.groupby("sid").agg(n=("pol", "size"), pols=("pol", set))
mixtas = por_sid.pols.apply(lambda s: {"positive", "negative"} <= s)
print(f"Oraciones con opinión: {len(por_sid)} | con más de un aspecto: {(por_sid.n > 1).mean():.1%} | "
      f"con positivo y negativo a la vez: {mixtas.mean():.1%}")
pol.sort_values("total", ascending=False)

<Figure size 800x420 with 1 Axes>

Oraciones con opinión: 2303 | con más de un aspecto: 37.2% | con positivo y negativo a la vez: 6.5%


pol,positive,neutral,negative,total,% negativas
cat,,,,,
FOOD#QUALITY,894,63,179,1136,15.8
RESTAURANT#GENERAL,592,26,144,762,18.9
SERVICE#GENERAL,527,20,179,726,24.7
AMBIENCE#GENERAL,265,21,133,419,31.7
FOOD#STYLE_OPTIONS,136,13,112,261,42.9
FOOD#PRICES,91,5,72,168,42.9
RESTAURANT#PRICES,66,17,71,154,46.1
DRINKS#QUALITY,34,0,7,41,17.1
DRINKS#STYLE_OPTIONS,16,1,23,40,57.5


La comida acapara el volumen (1,136 de las 3,792 opiniones son sobre su calidad), pero es de lo que menos se quejan: solo el 15.8% de esas opiniones son negativas. En servicio sube al 24.7%, en ambiente al 31.7% y en precios pasa del 40%.

Dos datos condicionan cómo se modela. El 37.2% de las oraciones con opinión habla de más de un aspecto, así que un clasificador por oración tiene que ser **multi-etiqueta**. Y el 6.5% elogia un aspecto y critica otro en la misma oración, así que la polaridad de la oración no alcanza para saber de qué se quejan. Por eso el detector de TripAdvisor predice falla **por etapa**.

### 2. Preprocesamiento: no hay que borrar el "no"

La lista de stopwords de NLTK incluye "no", "nunca", "sin" y "muy", que son justo las palabras que cambian la polaridad. Comparamos tres variantes con la misma representación y el mismo modelo (C = 1, sin ajustar).

In [3]:
variantes = modelos.variantes_stopwords(sem)
display(variantes)

pura = modelos.oraciones_polaridad(sem)
con_no = pura.texto.str.lower().str.contains(r"\bno\b")
print(f"Oraciones con 'no': {con_no.mean():.1%} | de ellas, negativas: {pura.y[con_no].mean():.1%} "
      f"(en el resto: {pura.y[~con_no].mean():.1%})")
ejemplo = "La comida no estaba mal, pero el servicio nunca llegó."
print("\nEjemplo:", ejemplo)
print("  sin negadores en stopwords:", lexico.tokenizador_es(True)(ejemplo))
print("  stopwords completas       :", lexico.tokenizador_es(False)(ejemplo))

,TF-IDF + LogReg,TF-IDF + NB,BoW + LogReg,BoW + NB
Variante,,,,
Stopwords sin negadores (pipeline final),0.837,0.780,0.819,0.828
Stopwords completas de NLTK,0.810,0.729,0.791,0.789
Sin quitar stopwords,0.827,0.746,0.813,0.827


Oraciones con 'no': 15.0% | de ellas, negativas: 66.9% (en el resto: 20.3%)

Ejemplo: La comida no estaba mal, pero el servicio nunca llegó.
  sin negadores en stopwords: ['comida', 'no', 'mal', 'pero', 'servicio', 'nunca', 'llegó']
  stopwords completas       : ['comida', 'mal', 'servicio', 'nunca', 'llegó']


Quitar las stopwords completas le cuesta a la regresión logística 2.7 puntos de F1 (0.837 contra 0.810) y a Naive Bayes 5.1 (0.780 contra 0.729). La razón está en el "no": aparece en el 15.0% de las oraciones, y el 66.9% de ellas son negativas, contra el 20.3% del resto. Dejar todas las stopwords tampoco es lo mejor (0.827), porque se suman palabras que no aportan.

Esta decisión se mantiene en inglés: el tokenizador de TripAdvisor conserva "not", "never", "didn't" y similares, y las redes no quitan ninguna stopword.

### 3. Colocaciones con PMI

El PMI mide si dos palabras aparecen juntas más de lo que daría el azar. Sirve para decidir qué bigramas valen como unidad ("valor seguro", "tourist trap") y, en el 03, para buscar candidatos al léxico en inglés.

In [4]:
textos_tok = [lexico.tokenizar(x) for x in sem.drop_duplicates("sid").texto]
lexico.pmi_bigramas(textos_tok).head(12).round(2).reset_index(drop=True)

,bigrama,frecuencia,PMI
0,estrella michelin,10,10.77
1,arroz caldoso,10,10.14
2,valor seguro,8,9.58
3,estuvimos cenando,8,9.09
4,varias veces,10,9.08
5,ha encantado,11,8.92
6,tal vez,9,8.68
7,desde luego,11,8.66
8,ha sido,26,8.62
9,hemos comido,12,8.59


### 4. Aspectos con palabras semilla y su expansión con embeddings

Una oración se asigna a una dimensión si contiene alguna de sus semillas, y se compara contra las categorías anotadas. Después probamos ampliar cada lista con sus vecinos en el espacio fastText SUC, y si eso ayuda.

In [5]:
from gensim.models import KeyedVectors

resultados = {"Solo semillas": lexico.evaluar_semillas(sem, lexico.SEMILLAS)}
ruta_suc = RAIZ / "Lab#9" / "embeddings-m-model.vec"
if ruta_suc.exists():
    suc = KeyedVectors.load_word2vec_format(ruta_suc, limit=100_000)
    conteo = collections.Counter(w for s in textos_tok for w in s)
    resultados["+ fastText SUC"] = lexico.evaluar_semillas(sem, {d: lexico.expandir_semillas(suc, s) for d, s in lexico.SEMILLAS.items()})
    filtrado = {d: lexico.expandir_semillas(suc, s, conteo_corpus=conteo) for d, s in lexico.SEMILLAS.items()}
    resultados["+ fastText, solo palabras del corpus"] = lexico.evaluar_semillas(sem, filtrado)
    for d, lex in filtrado.items():
        print(f"  {d:9}", ", ".join(lex[len(lexico.SEMILLAS[d]):][:12]))
else:
    print("No está Lab#9/embeddings-m-model.vec; se omite la expansión.")

tabla = pd.concat({k: v["F1"] for k, v in resultados.items()}, axis=1)
tabla.loc["Promedio"] = tabla.mean().round(3)
display(tabla)
resultados["Solo semillas"]

  Cocina    sopa, ensaladas, asado, puré, salsa, ternera, postres, tempura, jamón, queso, risotto, tarta
  Servicio  cocinero, maitre, cuidado, amigo, cliente, amable
  Ambiente  acogedor, comedor, hotel, restaurante, acogedora, cocina, bar
  Precio    iva, descuento


,Solo semillas,+ fastText SUC,"+ fastText, solo palabras del corpus"
Cocina,0.637,0.663,0.699
Servicio,0.854,0.859,0.858
Ambiente,0.679,0.600,0.580
Precio,0.809,0.802,0.803
Promedio,0.745,0.731,0.735


,precisión,cobertura,F1
Cocina,0.78,0.54,0.637
Servicio,0.95,0.78,0.854
Ambiente,0.76,0.61,0.679
Precio,0.76,0.86,0.809


Con las semillas solas, el F1 promedio es 0.745. Servicio es la dimensión más limpia: precisión 0.95 y cobertura 0.78. Ampliar las listas con fastText SUC no ayuda en promedio (0.731, o 0.735 si solo se aceptan palabras del corpus). Sí ayuda en cocina (de 0.637 a 0.699, porque agrega platos como sopa, ternera o risotto), pero empeora ambiente (de 0.679 a 0.580): el vecino de "local" en SUC trae "hotel", "restaurante" y "cocina", que no son ambiente.

Es la misma lección que repite el 03 en inglés: los vecinos en un espacio de embeddings son palabras **del mismo tema**, no de la misma categoría de error. Una lista ampliada por similitud gana cobertura y pierde precisión, y por eso el léxico de etiquetas débiles se revisa a mano.

### 5. Polaridad: Naive Bayes contra regresión logística

TF-IDF de unigramas y bigramas, con el pipeline de la sección 2. La C de la regresión logística se elige con validación cruzada agrupada por reseña, para que las oraciones de una misma reseña no queden a los dos lados. El test oficial se usa una sola vez.

In [6]:
r = modelos.polaridad_semeval(sem)
print("F1 macro en CV por C:", {k: round(v, 3) for k, v in r["barrido"].items()}, "→ C =", r["C"])
print(f"Regresión logística: F1 macro en test = {r['f1_logreg']:.3f}")
print(f"Naive Bayes        : F1 macro en test = {r['f1_nb']:.3f}")

B = r["vec"].transform(r["test"].texto)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a, (nombre, m) in zip(ax, [("Regresión logística", r["logreg"]), ("Naive Bayes", r["nb"])]):
    sns.heatmap(confusion_matrix(r["test"].y, m.predict(B)), annot=True, fmt="d", cmap="Blues", cbar=False, ax=a,
                xticklabels=["pos", "neg"], yticklabels=["pos", "neg"])
    a.set(title=nombre, xlabel="Predicho", ylabel="Real")
plt.tight_layout(); plt.savefig(FIGURAS / "02_confusion_polaridad.png", dpi=150); plt.show()

vocab = np.array(r["vec"].get_feature_names_out()); w = r["logreg"].coef_[0]; o = np.argsort(w)
print("Empujan a negativo:", ", ".join(vocab[o[::-1][:12]]))
print("Empujan a positivo:", ", ".join(vocab[o[:12]]))

F1 macro en CV por C: {0.1: 0.799, 0.3: 0.806, 1: 0.814, 3: 0.818, 10: 0.815, 30: 0.816} → C = 3
Regresión logística: F1 macro en test = 0.834
Naive Bayes        : F1 macro en test = 0.780


<Figure size 900x340 with 2 Axes>

Empujan a negativo: no, caro, poco, mejorar, peor, escaso, mal, desear, escasa, mas, euros, excepto
Empujan a positivo: muy, excelente, bien, buena, recomendable, comida, siempre, recomiendo, buen, perfecto, mejores, correcto


In [7]:
te = r["test"].assign(pred=r["logreg"].predict(B))
err = te[te.pred != te.y]
print(f"Errores de la regresión logística: {len(err)} de {len(te)}")
for _, x in err.sample(8, random_state=1).iterrows():
    print(f"  real={'neg' if x.y else 'pos'} pred={'neg' if x.pred else 'pos'} | {x.texto[:140]}")

Errores de la regresión logística: 83 de 611
  real=pos pred=neg | He estado muchas veces, lo he recomendado a familiares, amigos y allegados y nunca ha defraudado a nadie.
  real=neg pred=pos | Uno de estos dias mi experiencia fue del todo negativa puesto que al parecer uno de los dueños o encargado entro en la cocina y los gritos s
  real=neg pred=pos | Sólo un pequeño detalle, faltaba música de ambiente para ser perfecto.
  real=pos pred=neg | la verdad es que comimos realmente muy bien y no fue caro.
  real=neg pred=pos | Encima de mi cabeza tenía la pantalla del televisor con el sonido a mi oido izquierdo.
  real=pos pred=neg | Desde la dirección,pasando por cocina y camareros,el trato es excelente ,pero todo esto no serviría de nada sin que también,como es así.
  real=pos pred=neg | Nada más entrar me sorprendio, causandome una buena impresión.
  real=neg pred=pos | Muy turistico.


La regresión logística llega a 0.834 de F1 macro y Naive Bayes a 0.780. Con bigramas, las variables están correlacionadas, y Naive Bayes las cuenta como evidencia independiente. La C elegida en validación cruzada es 3, y el barrido es plano entre 1 y 30 (0.814 a 0.818).

Los pesos tienen sentido: "no", "caro", "poco" y "peor" empujan a negativo; "excelente", "recomendable" y "perfecto", a positivo. Los 83 errores de 611 muestran los límites de una bolsa de palabras:

- **Negación de lo negativo:** *"nunca ha defraudado a nadie"* y *"no fue caro"* salen negativas.
- **Quejas sin palabras de queja:** *"tenía la pantalla del televisor con el sonido a mi oido izquierdo"*.
- **Contexto:** *"Muy turistico."* es negativo solo si uno sabe lo que implica.

Estos son los casos que motivan, en TripAdvisor, un modelo que lea la oración en orden (LSTM).

### Cifras clave


In [8]:
resultados.guardar("02", {
    "f1_logreg": r["f1_logreg"], "f1_nb": r["f1_nb"], "C": r["C"], "variantes_stopwords": variantes,
    "pct_con_no": con_no.mean() * 100, "pct_neg_con_no": pura.y[con_no].mean() * 100, "pct_neg_sin_no": pura.y[~con_no].mean() * 100,
    "semillas_f1": tabla, "multiaspecto_pct": (por_sid.n > 1).mean() * 100, "mixtas_pct": mixtas.mean() * 100,
    "errores_logreg": len(err), "test_n": len(te),
})

hallazgos_02.json: 12 claves → C, errores_logreg, f1_logreg, f1_nb, mixtas_pct, multiaspecto_pct, pct_con_no, pct_neg_con_no, pct_neg_sin_no, semillas_f1, test_n, variantes_stopwords


PosixPath('/Users/georgealbadr/Github/Natural_Language_Processing/resultados/hallazgos_02.json')

### Hallazgos

- **Polaridad en español:** la regresión logística llega a 0.834 de F1 macro y Naive Bayes a 0.780 en el test oficial de SemEval. Reproduce el avance.
- **Conservar los negadores vale 2.7 puntos de F1** (0.837 contra 0.810): el 66.9% de las oraciones con "no" son negativas. Ningún tokenizador del proyecto quita los negadores.
- **El 37.2% de las oraciones habla de más de un aspecto y el 6.5% mezcla polaridades:** el detector tiene que ser multi-etiqueta y por etapa.
- **Las semillas dan un F1 de 0.745 (servicio 0.854, con precisión 0.95), y ampliarlas con fastText no mejora el promedio (0.731):** los embeddings agrupan por tema, no por tipo de falla.
- **Los errores de la bolsa de palabras son negaciones de lo negativo (*"nunca ha defraudado a nadie"*) y quejas sin palabras de queja,** justo lo que debería capturar un modelo secuencial.

### Verificación de citas

Toda oración citada entre comillas en este notebook tiene que existir literalmente en el corpus.

In [9]:
citas = ['Muy turistico.', 'no fue caro', 'nunca ha defraudado a nadie', 'tenía la pantalla del televisor con el sonido a mi oido izquierdo']
verificadas = datos.verificar_citas(citas, ('semeval',))
print(f"{len(verificadas)} citas verificadas:", verificadas)

4 citas verificadas: {'Muy turistico.': 'semeval', 'no fue caro': 'semeval', 'nunca ha defraudado a nadie': 'semeval', 'tenía la pantalla del televisor con el sonido a mi oido izquierdo': 'semeval'}
